# PySide laser widget

Requires an editable install from the Laser repo root:

```bash
pip install -e ".[pyside,notebooks]"
```

Enable Qt in this kernel before creating widgets (`%gui qt`).

Two ways to start the panel:

1. **Normal launch** — `LaserControlWidget()` scans USB; you Connect in the GUI.
2. **Programmatic** — `TLB8800.connect(port)` then `LaserControlWidget(laser)` starts already connected. `widget.laser` is the same object.

In [1]:
%gui qt

## 1. Launch like the standalone app

No laser instance is passed. The widget discovers COM ports; use **Connect** in the panel.

In [7]:
from laser.ui.pyside import LaserControlWidget

widget = LaserControlWidget()
widget.resize(1100, 900)
widget.show()
widget

<laser.ui.pyside.widget.LaserControlWidget(0x2506c5bb240) at 0x000002506DE62340>

## 2. Discover, connect, pass `TLB8800`

Close the previous widget (or skip cell 1) so the COM port is free. After connect, the device menu should show this laser as connected. Notebook `laser` and `widget.laser` are the same session. Widget **Disconnect** closes the port so **Refresh** can find it again (`laser.is_open` is then `False`).

In [2]:
from laser.discovery import LaserDiscovery
from laser.newfocus import TLB8800
from laser.ui.pyside import LaserControlWidget

found = LaserDiscovery().discover()
print(found)


{'COM3': 'New Focus;TLM8700;TLB-8800-0056;2.6.0;2122'}


In [3]:
port = next(iter(found))
print(port)

COM3


In [4]:
laser = TLB8800.connect(port)
print("open:", laser.is_open, "port:", laser.port)


open: True port: COM3


In [5]:
laser.is_open

True

In [6]:

widget = LaserControlWidget(laser)
widget.resize(1100, 900)
widget.show()

print("same object:", widget.laser is laser)
laser.read.current()

same object: True


38.6

## 3. Programmatic sets, then sync the panel

Lock the setpoint cells so a click cannot fight the notebook. After `laser.set.*`, pull instrument state back into the GUI. Unlocking also syncs.

In [7]:
widget.remotecontrol(False)

In [ ]:
gui = widget
gui.remotecontrol(True)
laser.set.current(80)
print("instrument current:", laser.read.current())
gui.sync_gui_panel_to_laser()
gui.remotecontrol(False)